# 🛰️ 01 - SAR Oil Spill Dataset Exploration & Class Analysis

This notebook explores Sentinel-1 SAR imagery and 5-class semantic masks from the **Krestenitis et al. (2019)** dataset benchmark.

### Semantic Classes:
- **0: Sea Surface** (Background / ocean backscatter)
- **1: Oil Spill** (Primary positive class - dark wave-damped slicks)
- **2: Look-alike** (Confounding non-oil dark phenomena: low wind, biogenic films)
- **3: Ship** (Bright corner reflectors / metallic vessels)
- **4: Land** (Coastal landmasses / high rough surface return)

In [ ]:
import sys
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# Add src to path
sys.path.append(str(Path.cwd().parent / "src"))

from oilspill.data.dataset import CLASS_NAMES, CLASS_COLORS, rgb_to_mask, mask_to_rgb
from oilspill.data.validation import DatasetAuditor
from oilspill.utils.io import load_json

## 1. Load Dataset Audit Statistics

In [ ]:
audit_path = Path("../artifacts/reports/dataset_audit_report.json")
if audit_path.exists():
    report = load_json(audit_path)
    print(f"Total Scene Pairs: {report['total_pairs_analyzed']}")
    print(f"Image Dimensions: {report['dimensions_found']}")
    
    df_dist = pd.DataFrame([
        {
            "Class ID": i,
            "Class Name": CLASS_NAMES[i],
            "Pixel Count": report['class_pixel_counts'][CLASS_NAMES[i]],
            "Percentage": f"{report['class_pixel_percentages'][CLASS_NAMES[i]]:.4f}%",
            "Image Presence": f"{report['class_image_presence_percentage'][CLASS_NAMES[i]]:.2f}%",
            "Suggested Weight": f"{report['suggested_class_weights'][CLASS_NAMES[i]]:.2f}",
        }
        for i in range(5)
    ])
    display(df_dist)
else:
    print("Audit report not found. Run 'python scripts/prepare_data.py' first.")

## 2. Visualize Sample SAR Scenes, Masks & Overlays

In [ ]:
raw_dir = Path("../data/raw")
auditor = DatasetAuditor(raw_dir)
img_paths, msk_paths, _ = auditor.scan_pairs()

if len(img_paths) > 0:
    num_samples = min(3, len(img_paths))
    fig, axes = plt.subplots(num_samples, 3, figsize=(16, 4 * num_samples))
    
    if num_samples == 1:
        axes = np.expand_dims(axes, 0)
        
    for i in range(num_samples):
        img = cv2.imread(str(img_paths[i]), cv2.IMREAD_GRAYSCALE)
        msk_raw = cv2.imread(str(msk_paths[i]), cv2.IMREAD_UNCHANGED)
        
        if msk_raw.ndim == 3:
            msk = rgb_to_mask(cv2.cvtColor(msk_raw, cv2.COLOR_BGR2RGB))
        else:
            msk = np.clip(msk_raw, 0, 4)
            
        msk_rgb = mask_to_rgb(msk)
        sar_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        overlay = cv2.addWeighted(sar_rgb, 0.65, msk_rgb, 0.35, 0)
        
        axes[i, 0].imshow(img, cmap="gray")
        axes[i, 0].set_title(f"SAR Scene: {img_paths[i].name}")
        axes[i, 0].axis("off")
        
        axes[i, 1].imshow(msk_rgb)
        axes[i, 1].set_title("5-Class Ground Truth Mask")
        axes[i, 1].axis("off")
        
        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title("Overlay")
        axes[i, 2].axis("off")
        
    legend_patches = [mpatches.Patch(color=np.array(CLASS_COLORS[c])/255.0, label=f"{c}: {CLASS_NAMES[c]}") for c in range(5)]
    fig.legend(handles=legend_patches, loc="upper center", bbox_to_anchor=(0.5, 1.02), ncol=5, fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("No images found in data/raw. Run 'python scripts/download_data.py --synthetic' to create sample scenes.")